# Baseline COPV Validation

This notebook is the thin baseline validation entrypoint for the packaged COPV optimizer. It builds the shell,
meshes it, assembles the JAX FEM state, solves the baseline case, and exports a baseline VTU artifact.


## Setup

The notebook imports reusable code from `src/copv_opt/` instead of carrying engineering logic inline.


In [ ]:
from jax import config as jax_config
ENABLE_X64 = True
jax_config.update("jax_enable_x64", ENABLE_X64)

import json
import sys
from pathlib import Path

import jax
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Image, display


def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
            return candidate
    raise FileNotFoundError("Could not locate the project root containing pyproject.toml and src/")


PROJECT_ROOT = find_project_root(Path.cwd())
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

OUTPUTS = PROJECT_ROOT / "outputs"
OUTPUTS.mkdir(parents=True, exist_ok=True)

print(f"Project root : {PROJECT_ROOT}")
print(f"Outputs      : {OUTPUTS}")
print(f"JAX backend  : {jax.default_backend()}")
print(f"JAX devices  : {jax.devices()}")
print(f"JAX x64      : {ENABLE_X64}")

from copv_opt.config import GeometryConfig, MaterialConfig
from copv_opt.geometry import ensure_copv_mesh
from copv_opt.physics import baseline_response, build_copv_fem_state, make_solve_compliance
from copv_opt.visualize import set_copv_axes, show_copv_mesh, write_vtu


## Geometry and Meshing


In [ ]:
geom = GeometryConfig()
material = MaterialConfig()

step_path = OUTPUTS / "copv_shell.step"
msh_path = OUTPUTS / "copv_shell.msh"
mesh = ensure_copv_mesh(step_path, msh_path, geom, remesh=True)
state = build_copv_fem_state(mesh.nodes, mesh.elems, material, geom)
solve_compliance = make_solve_compliance(state)

print(f"STEP file    : {step_path}")
print(f"Mesh file    : {msh_path}")
print(f"Mesh nodes   : {len(mesh.nodes)}")
print(f"Mesh tetra   : {len(mesh.elems)}")
print(f"Free DOFs    : {len(np.asarray(state['free_dofs']))}")

fig = show_copv_mesh(
    mesh.nodes,
    state["outer_faces"],
    geom,
    f"COPV analysis mesh: {len(mesh.nodes)} nodes / {len(mesh.elems)} tetra",
    OUTPUTS / "copv_analysis_mesh.png",
)
display(fig)
plt.close(fig)


## Baseline Solve


In [ ]:
baseline = baseline_response(state, material, solve_compliance)
base_u = np.asarray(baseline["displacement"]).reshape(len(mesh.nodes), 3)
u_mag = np.linalg.norm(base_u, axis=1)

fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection="3d")
ax.scatter(mesh.nodes[:, 0], mesh.nodes[:, 1], mesh.nodes[:, 2], c=u_mag, cmap="plasma", s=5)
ax.set_title("Baseline displacement magnitude")
set_copv_axes(ax, geom.outer_radius, geom.cylinder_length)
ax.view_init(20, 36)
fig.tight_layout()
display(fig)
fig.savefig(OUTPUTS / "baseline_displacement.png", dpi=115)
plt.close(fig)

base_vtu = write_vtu(
    OUTPUTS / "copv_base.vtu",
    mesh.nodes,
    mesh.elems,
    base_u,
    np.asarray(baseline["thickness"]),
    np.asarray(baseline["density"]),
    np.asarray(baseline["fiber_dirs"]),
    np.asarray(baseline["coverage"]),
)

summary = {
    "jax_backend": jax.default_backend(),
    "mesh": {
        "nodes": int(len(mesh.nodes)),
        "elements": int(len(mesh.elems)),
        "step": str(step_path),
        "msh": str(msh_path),
    },
    "baseline": {
        "strain_energy": float(np.asarray(baseline["compliance"])),
        "mass_metric": float(np.asarray(baseline["mass_metric"])),
        "vtu": str(base_vtu),
    },
}

summary_path = OUTPUTS / "baseline_summary.json"
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(json.dumps(summary, indent=2))


## Interactive Viewer Hooks

The package exposes interactive PyVista viewers from `copv_opt.visualize` for local inspection of the exported VTU file.


In [ ]:
# Uncomment locally to inspect the baseline VTU in an interactive window.
#
# from copv_opt.visualize import render_vtu_interactive
# render_vtu_interactive(OUTPUTS / "copv_base.vtu", scalar_field="displacement_norm", slice_model=True)
